# 00 — Code Walkthrough

A guided tour of every `src` module: **what it does, why it exists, and a runnable example.**
Read this to understand the codebase without digging through files.

> **The golden rule of this project:** deterministic code computes the exact numbers (the truth);
> the LLM only *explains* them. Watch for the `query_engine` — that's the truth layer everything
> else is built around.

**Kernel:** Python 3.12. Restart & Run All.

In [1]:
# bootstrap: find the project's packages + dataset regardless of where this runs
import sys, importlib.util
from pathlib import Path
if importlib.util.find_spec('pandas') is None:
    _tag = f'python{sys.version_info.major}.{sys.version_info.minor}'
    for _b in [Path.cwd(), *Path.cwd().parents]:
        _sp = _b / '.venv' / 'lib' / _tag / 'site-packages'
        if _sp.exists():
            sys.path.insert(0, str(_sp)); break
import numpy as np, pandas as pd
def find_repo():
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / 'data' / 'jena_climate_2009_2016.csv').exists():
            return base
    raise FileNotFoundError('Run from inside the repo.')
ROOT = find_repo(); sys.path.insert(0, str(ROOT / 'v1'))
from src import (preprocess, analysis, anomaly, forecasting, deep as da,
                 deep as dl, clustering, query_engine as qe, reasoning,
                 rag_system, llm, interface, evaluation)
print('All src modules imported. Deep-model device:', da.get_device())

All src modules imported. Deep-model device: cuda


## 1. `preprocess` — get clean, regular data
Loads the CSV, parses timestamps, removes the `-9999` sensor-error sentinels, interpolates gaps,
and resamples everything to a common **hourly** frequency. Key calls: `load_data`, `clean`, `align`.

In [2]:
raw = preprocess.load_data(ROOT / 'data' / 'jena_climate_2009_2016.csv')
df = preprocess.align(preprocess.clean(raw), freq='1h')
print(f'{len(raw):,} raw rows -> {len(df):,} hourly rows, {df.shape[1]} sensors')
SENSOR = 'T (degC)'
df.head(3)

420,551 raw rows -> 70,129 hourly rows, 14 sensors


,p (mbar),T (degC),Tpot (K),Tdew (degC),rh (%),VPmax (mbar),VPact (mbar),VPdef (mbar),sh (g/kg),H2OC (mmol/mol),rho (g/m**3),wv (m/s),max. wv (m/s),wd (deg)
Date Time,,,,,,,,,,,,,,
2009-01-01 00:00:00,996.528,-8.304000,265.118000,-9.120000,93.780000,3.260000,3.058000,0.202000,1.910000,3.068000,1309.196000,0.520000,1.002000,174.460000
2009-01-01 01:00:00,996.525,-8.065000,265.361667,-8.861667,93.933333,3.323333,3.121667,0.201667,1.951667,3.133333,1307.981667,0.316667,0.711667,172.416667
2009-01-01 02:00:00,996.745,-8.763333,264.645000,-9.610000,93.533333,3.145000,2.940000,0.201667,1.836667,2.950000,1311.816667,0.248333,0.606667,196.816667


## 2. `analysis` — describe how the data behaves
Rolling statistics, **STL decomposition** (trend + seasonal + residual), and correlations.
The *residual* (what's left after removing trend and season) is where anomalies hide.

In [3]:
decomp = analysis.decompose(df[SENSOR], period=24)   # 24 hourly steps = one daily cycle
print('STL components:', list(decomp.columns))
print('\nTop correlations with temperature:')
print(analysis.top_correlations(df, SENSOR, n=3))

STL components: ['observed', 'trend', 'seasonal', 'resid']

Top correlations with temperature:
Tpot (K)        0.996815
rho (g/m**3)   -0.963304
VPmax (mbar)    0.951173
Name: T (degC), dtype: float64


## 3. `query_engine` — the truth layer 
Deterministic functions that compute **exact** answers (and *when* they happened). The LLM is never
allowed to do this. Everything returns structured results so the explainer has all it needs.

In [4]:
print('Sensor summary (exact):')
for k, v in qe.sensor_summary(df, SENSOR).items():
    print(f'  {k:>8}: {v}')
# prove it matches pandas exactly
assert qe.get_max(df, SENSOR).value == df[SENSOR].max()
print('\nVerified: query engine == pandas (exact).')

Sensor summary (exact):
    sensor: T (degC)
      mean: 9.444727092215773
       std: 8.409740076707907
       min: -22.653333333333336
       max: 37.038333333333334
     range: 59.69166666666667
   missing: 0
    max_at: 2015-07-04 16:00:00
    min_at: 2009-01-07 03:00:00

Verified: query engine == pandas (exact).


## 4. `anomaly` — statistical + classical-ML detectors
`zscore_anomalies` flags points far from normal (run on the STL residual so season is removed).
`isolation_forest_anomalies` flags unusual *multi-sensor combinations*.

In [5]:
z = anomaly.zscore_anomalies(decomp['resid'].dropna(), threshold=4.0)
iso = anomaly.isolation_forest_anomalies(df, contamination=0.01)
print(f'z-score flags: {int(z.sum())}   Isolation Forest flags: {int(iso.sum())}')
print('First anomaly windows:', anomaly.anomaly_windows(z)[:2])

z-score flags: 526   Isolation Forest flags: 702
First anomaly windows: [(Timestamp('2009-01-07 03:00:00'), Timestamp('2009-01-07 13:00:00')), (Timestamp('2009-01-14 07:00:00'), Timestamp('2009-01-14 09:00:00'))]


## 5. `forecasting` — ARIMA forecast-error detector
Fits ARIMA (after removing the seasonal cycle) and flags points the model fails to predict.

In [6]:
import warnings; warnings.filterwarnings('ignore')
arima = forecasting.arima_anomalies(df[SENSOR], order=(2,1,2), threshold=4.0)
print(f'ARIMA flags: {int(arima.sum())}')

ARIMA flags: 411


## 6. `deep` — deep detectors (GPU-aware)
Unsupervised neural nets that learn the *normal* shape and flag what they can't reproduce.
`deep` = 1D CNN autoencoder + LSTM autoencoder + LSTM forecaster (all in one module).

In [7]:
cnn = da.cnn_ae_anomalies(df[SENSOR], window=24, epochs=10, quantile=0.99)
lstm = dl.lstm_ae_anomalies(df[SENSOR], window=24, epochs=8, quantile=0.99)
print(f'CNN-AE flags: {int(cnn.sum())}   LSTM-AE flags: {int(lstm.sum())}')

CNN-AE flags: 702   LSTM-AE flags: 702


## 7. `clustering` — group similar days
Summarises each day into a feature vector and K-means-clusters them. The clusters come out ordered
by temperature — i.e. it rediscovers the seasons.

In [8]:
feats = clustering.daily_features(df)
labels = clustering.kmeans_cluster(feats, k=4)
profiles = clustering.cluster_profiles(feats, labels)[f'mean_{SENSOR}'].round(1).sort_values()
print('Cluster mean temperatures (cold -> warm):', list(profiles.values))

Cluster mean temperatures (cold -> warm): [np.float64(0.5), np.float64(6.5), np.float64(12.2), np.float64(19.6)]


## 8. `reasoning` — explain *why* a moment is anomalous
Looks at every sensor's z-score at a timestamp to describe what was unusual (multi-sensor reasoning).

In [9]:
hottest = qe.get_max(df, SENSOR).when
print(reasoning.describe_moment(df, hottest))

At 2015-07-04 16:00:00: VPdef (mbar) unusually high (+8.4σ); VPmax (mbar) unusually high (+6.4σ); T (degC) unusually high (+3.3σ); Tpot (K) unusually high (+3.2σ).


## 9. `rag_system` — local context retrieval + trust-aware template
A TF-IDF search over short sensor descriptions (the 'R' in RAG), plus `explain()` which formats an
answer with trust labels (computed / context / inferred). Fully offline, no LLM needed.

In [10]:
ret = rag_system.Retriever()
print('Retrieved for "how humid is it":', [s.sensor for s in ret.retrieve('how humid is it', k=2)])
print()
print(rag_system.explain('Q: hottest?', computed='max T was 37.04 at 2015-07-04.',
                         context='T (degC) is air temperature.'))

Retrieved for "how humid is it": []

Q: hottest?
🔢 Computed: max T was 37.04 at 2015-07-04.
📚 Context: T (degC) is air temperature.


## 10. `llm` — optional local Ollama explainer
Rephrases computed facts into natural language **without changing numbers**. Optional: if Ollama
isn't running, the system uses the template above instead.

In [11]:
print('Ollama available:', llm.is_available())
print('Models:', llm.list_models() or '(none)')

Ollama available: True
Models: ['llama3.2:3b', 'hf.co/Andycurrent/Gemma-3-1B-it-GLM-4.7-Flash-Heretic-Uncensored-Thinking_GGUF:F16', 'hf.co/HauhauCS/Gemma-4-E4B-Uncensored-HauhauCS-Aggressive:Q6_K_P']


## 11. `interface` — the natural-language front door
Maps a plain question to a query-engine call, attaches context, and returns a trust-aware answer.
Keyword routing first; the LLM only interprets questions the keywords miss (numbers still from code).

In [12]:
iface = interface.Interface(df)            # use_llm=False -> template phrasing
for q in ['what was the hottest temperature?', 'how does temperature relate to density?']:
    print(iface.ask(q).text)
    print('-'*60)

Q: what was the hottest temperature?
🔢 Computed: max T (degC) was 37.04 at 2015-07-04 16:00:00.
📚 Context: T (degC) is the air temperature in degrees Celsius. Typical range about -23 to 37 C, with strong seasonal and daily cycles.
------------------------------------------------------------
Q: how does temperature relate to density?
🔢 Computed: correlation(T (degC), rho (g/m**3)) = -0.963.
🧠 Inferred: They move in opposite directions.
------------------------------------------------------------


## 12. `evaluation` — measure detector quality
Because the data has no anomaly labels, we inject *known* spikes and score detectors against them
(`inject_spikes`, `score`, `confusion`).

In [13]:
noisy, truth = evaluation.inject_spikes(df[SENSOR], n=50, magnitude=6.0)
pred = anomaly.zscore_anomalies(analysis.decompose(noisy, period=24)['resid'].dropna(), threshold=4.0)
print('confusion:', evaluation.confusion(pred, truth))
sc = evaluation.score(pred, truth)
print(f'precision={sc.precision:.2f} recall={sc.recall:.2f} f1={sc.f1:.2f}')

confusion: {'TP': 21, 'FP': 255, 'FN': 29, 'TN': 69824}
precision=0.08 recall=0.42 f1=0.13


## Recap
- `preprocess` → `analysis` → (`anomaly`, `forecasting`, `deep`) → `clustering`
  build the analytics.
- **`query_engine`** is the exact truth layer.
- `reasoning`, `rag_system`, `llm`, `interface` turn results into trustworthy natural-language answers.
- `evaluation` measures how good the detectors are.

Next: `01_data_exploration` for the data, or `08_model_comparison` for the full results.